# Day 5 — Solution: Errors, Power, Effect Size

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
from scipy import stats
from scipy.stats import norm

## E1 — power three ways

In [ ]:
rng = np.random.default_rng(0)
for Y in [1, 3, 5, 10, 25]:
    n = int(Y*252)
    sr_d = 0.5/np.sqrt(252)
    analytic = norm.cdf(0.5*np.sqrt(Y) - 1.645)
    x = rng.normal(sr_d, 1/np.sqrt(252), (2000, n))
    t = x.mean(axis=1)*np.sqrt(n)
    sim = (t > 1.645).mean()
    print(f"Y={Y:2d}: analytic {analytic:.3f} | simulated {sim:.3f}")

**Expected agreement:** 0.126 / 0.218 / 0.299 / 0.475 / 0.804 —
analytic and simulated match within simulation error. **A real SR-0.5
strategy is more likely to be MISSED than caught at 5 years (70/30);
the "we'll know in five years" plan has the evidence-weight of one
coin flip.**

## E2 — the 2×2, priced

In [ ]:
alpha, power_ = 0.05, 0.30
p_null, p_real = 0.9, 0.1
cost = p_null*alpha*2e6 + p_real*(1-power_)*8e6
print(f"expected cost per decision: ${cost:,.0f}")
# minimize over alpha: d/dalpha [0.9*alpha*2 + 0.1*(1-power(alpha))*8]
# power(alpha) falls as alpha falls... approximate by scanning:
for a in [0.01, 0.02, 0.05, 0.10, 0.20]:
    z = norm.ppf(1-a)
    # power at horizon where power(5%)=30%: SR*sqrt(Y) such that Phi(x-1.645)=.3 -> x=1.023
    pw = norm.cdf(1.023 - z)
    c = 0.9*a*2 + 0.1*(1-pw)*8
    print(f"alpha={a:.2f}: power={pw:.2f}, cost=${c:.2f}M")

**Expected reasoning.** At α=5%: ~$0.55M per decision. Scanning:
lowering α raises the miss cost faster than it saves the false-fund
cost (the reals are expensive at this shop) — the optimum sits at
α ≈ 10–20% *given* these economics and a 10% real-rate: **the
"right" significance level is a business parameter, not a
superstition** — the 5% convention is a choice somebody else made
about somebody else's cost matrix.

## E3 — posterior discounts

In [ ]:
def posterior(base, power_=0.6, alpha=0.05):
    return base*power_/(base*power_ + (1-base)*alpha)
for base in [0.2, 0.1, 0.02]:
    print(f"base 1/{1/base:.0f}: P(real|sig) = {posterior(base):.0%}")
# flip: what base gives 90%?
from scipy.optimize import brentq
b = brentq(lambda x: posterior(x) - 0.9, 1e-4, 0.99)
print(f"base for 90% posterior: 1 in {1/b:.0f}")

**Expected numbers.** 73% / 57% / 20%; a 90% posterior needs a base
rate of 1-in-4.6 — **the shop would have to be right about a quarter
of its ideas BEFORE looking at results.** That's the audit: if the
true hit rate is 1-in-20, most "significant" results are the noise
casino's house winnings.

## E4 — designing the study

In [ ]:
delta, sigma = 0.0003, 0.011
n = ((1.645+0.84)*sigma/delta)**2
print(f"single strategy: n = {n:,.0f} days = {n/252:.0f} years")
sigma_p = sigma/np.sqrt(30)
n_p = ((1.645+0.84)*sigma_p/delta)**2
print(f"30-edge portfolio: n = {n_p:,.0f} days = {n_p/252:.1f} years")

Single: ~8,400 days ≈ 33 years — dead design. Portfolio: 281 days ≈
1.1 years — alive. **Diversification divided σ by √30 and the
required n by 30: it is a data multiplier.** This is THE argument for
portfolio-level evaluation of signal families (and why shops test
"families of edges," not single edges — module 10's territory). The
fine print: the 30 edges must be genuinely independent (common factor
exposure doesn't divide), and each edge's δ must survive at portfolio
weights.

## E5 — the huge-t false positive (exemplar)

(1) **Selection**: t = 6.8 is the best of hundreds of tried variants —
the extreme of a batch, not a sample (module 02.17; the expected max
of 100 null t's is ~3, of 10,000 is ~4). (2) **Broken SE**: smoothed
or overlapping data (marked-to-model positions, 20-day averaging)
understates the SE — an autocorrelated series with the same mean can
multiply t by √(2H−1) without any additional information. (3) **Data
pathologies**: survivorship (the tested universe excludes the dead),
backfill (the good history added after the fact), or outright
look-ahead. A big t is a reason to audit the SE and the search, not a
reason to celebrate — the strongest results deserve the most
suspicion because every bias in the book inflates t.